# Prerequisites

## Read All ecommerce.silver tables in dfs

In [0]:
# customers_df = spark.read.table("ecommerce.silver.customers")
orders_df = spark.read.table("ecommerce.silver.orders")
order_items_df = spark.read.table("ecommerce.silver.order_items")
# products_df = spark.read.table("ecommerce.silver.products")
# payments_df = spark.read.table("ecommerce.silver.payments")
# inventory_df = spark.read.table("ecommerce.silver.inventory")
# clickstream_df = spark.read.table("ecommerce.silver.clickstream")


## Read order_items

In [0]:
order_items_df.limit(5).display()

In [0]:
order_items_df.printSchema()

# Calculations

## Join Orders and order_items

In [0]:
orders_order_items_df = orders_df.alias("o").join(order_items_df.alias("oi"), "order_id")
orders_order_items_df.limit(5).display()

## Calculate daily sales

In [0]:
from pyspark.sql.functions import *
orders_order_items_df.select("order_status").distinct().show()
orders_order_items_df.groupBy(col("order_status")).agg(countDistinct("order_id")).display()

In [0]:
from pyspark.sql.functions import *

daily_sales_df = (
    orders_order_items_df.withColumn("sales_date", to_date(col("order_date")))
    .groupBy("sales_date")
    .agg(
        round(sum("net_amount"), 2).alias("total_sales"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct(
            when(col("order_status") == lit("DELIVERED"), col("order_id"))
        ).alias("completed_orders"),
        countDistinct(
            when(col("order_status") == lit("CANCELLED"), col("order_id"))
        ).alias("cancelled_orders"),
        countDistinct(
            when(col("order_status") == lit("RETURNED"), col("order_id"))
        ).alias("returned_orders"),
        round(sum(col("gross_amount")), 2).alias("gross_sales"),
        round(sum(col("discount_amount")), 2).alias("discount_amount"),
        round(sum(when(col("order_status") != lit("RETURNED"), col("net_amount"))), 2).alias("net_sales"),
    )
    .sort("sales_date")
)
daily_sales_df.limit(5).display()

In [0]:
daily_sales_df = daily_sales_df.withColumn(
    "average_order_value",
    round(
        when(col("completed_orders") > 0, col("net_sales") / col("completed_orders")), 2
    ),
)
daily_sales_df.limit(5).display()

# Create Delta Table

In [0]:
%sql
use catalog ecommerce;
create schema if not exists gold;


In [0]:
daily_sales_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.gold.daily_sales")
spark.read.table("ecommerce.gold.daily_sales").display()

# Buisness Questions

## What was today's revenue?


In [0]:
today_revenue_df = spark.table("gold.daily_sales").filter(col("sales_date") == current_date()).select(
    "net_sales"
)
today_revenue_df.show()

## What is the average order value?

In [0]:
avg_order_val_df = (
    spark.read.table("gold.daily_sales")
    .select(round(avg(col("average_order_value")), 2)).alias("avg_order_value")
)
display(avg_order_val_df)

## Which day generated the highest revenue?

In [0]:
from pyspark.sql.window import Window

window_spec = Window.orderBy(col("net_sales").desc())
spark.read.table("ecommerce.gold.daily_sales").withColumn(
    "rank", dense_rank().over(window_spec)
).filter(col("rank") == 1).select(col("sales_date")).show()

## What is the cancellation rate?

In [0]:
spark.read.table("ecommerce.gold.daily_sales").withColumn(
    "cancellation_rate",
    round(((col("cancelled_orders") / col("total_orders")) * 100), 2),
).show()